# Synthetic Data Generation with LLMs

## Why Synthetic Data?
- Real labeled data is expensive and scarce
- Fine-tuning requires thousands of examples
- LLMs can generate diverse, high-quality data at scale
- Privacy-sensitive domains (medical, legal) can't use real data

## Methods Overview

| Method | Description | Key Paper |
|--------|-------------|----------|
| Self-Instruct | Use LLM to generate instructions from seeds | Wang et al. 2022 |
| Alpaca | GPT-3.5 generates 52K instruction pairs | Taori et al. 2023 |
| Evol-Instruct | Evolve simple instructions into complex ones | Xu et al. 2023 |
| Orca | Learn from GPT-4's reasoning traces | Mukherjee et al. 2023 |
| Magpie | Decode system prompt to get instruction templates | Xu et al. 2024 |
| Persona-driven | Generate from diverse user personas | Chan et al. 2024 |

## Self-Instruct Pipeline

```
1. Start with 175 seed tasks
2. Sample few-shot seeds
3. Generate new instruction with LLM
4. Filter low-quality (ROUGE > 0.7 → too similar)
5. Generate input/output for each instruction
6. Add to pool, repeat
```

In [1]:
# Self-Instruct style generation
from openai import OpenAI
import json
import random

client = OpenAI()

SEED_TASKS = [
    {"instruction": "Summarize the following article in 3 bullet points.", "category": "summarization"},
    {"instruction": "Translate the following sentence to French.", "category": "translation"},
    {"instruction": "Write a Python function to sort a list.", "category": "coding"},
    {"instruction": "Explain the concept of recursion.", "category": "explanation"},
    {"instruction": "What are the pros and cons of remote work?", "category": "opinion"},
]

def generate_new_instructions(seeds, n=5):
    sample = random.sample(seeds, min(3, len(seeds)))
    examples = "\n".join(f"- {s['instruction']}" for s in sample)

    prompt = f"""Here are some example instructions:
{examples}

Generate {n} new creative instructions that are diverse and different from these examples.
Return as JSON: {{"instructions": ["...", "...", ...]}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)["instructions"]

def generate_response(instruction):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Respond concisely."},
            {"role": "user", "content": instruction}
        ]
    )
    return response.choices[0].message.content

print('Self-Instruct pipeline functions defined.')
print('Seed tasks:', len(SEED_TASKS))

Self-Instruct pipeline functions defined.
Seed tasks: 5


In [2]:
# Evol-Instruct make simple instructions more complex
def evol_instruct(instruction, evolution_type="depth"):
    if evolution_type == "depth":
        prompt = f"""Make this instruction more complex and challenging by adding constraints,
requirements, or deeper reasoning steps. Keep it as a single instruction.

Original: {instruction}
Evolved:"""
    elif evolution_type == "breadth":
        prompt = f"""Create a completely new instruction inspired by this one but in a different domain.

Original: {instruction}
New instruction:"""
    elif evolution_type == "reasoning":
        prompt = f"""Modify this instruction to require multi-step reasoning or planning.

Original: {instruction}
Modified:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

# Example
original = "Write a function to reverse a string."
print(f'Original: {original}')
# evolved = evol_instruct(original, "depth")
# print(f'Evolved:  {evolved}')
print('(Run with API key to see evolved instruction)')

Original: Write a function to reverse a string.
(Run with API key to see evolved instruction)


In [3]:
# Quality filtering ROUGE-based deduplication
from rouge_score import rouge_scorer

def is_too_similar(new_instruction, existing_instructions, threshold=0.7):
    """Filter out instructions that are too similar to existing ones (ROUGE-L > threshold)"""
    scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    for existing in existing_instructions:
        score = scorer_obj.score(existing, new_instruction)['rougeL'].fmeasure
        if score > threshold:
            return True
    return False

# Test deduplication
existing = ["Write a function to reverse a string in Python."]
candidate1 = "Write a Python function that reverses a string."  # too similar
candidate2 = "Implement a binary search algorithm in JavaScript."  # different enough

print(f'Candidate 1 too similar: {is_too_similar(candidate1, existing)}')
print(f'Candidate 2 too similar: {is_too_similar(candidate2, existing)}')

Candidate 1 too similar: True
Candidate 2 too similar: False


In [4]:
# Distilabel industrial-grade synthetic data library
DISTILABEL_CODE = '''
# pip install distilabel[openai]
from distilabel.pipeline import Pipeline
from distilabel.steps import LoadDataFromHub
from distilabel.steps.tasks import TextGeneration, SelfInstruct
from distilabel.llms import OpenAILLM

with Pipeline(name="synthetic-data") as pipeline:
    load = LoadDataFromHub(repo_id="argilla/databricks-dolly-15k", split="train[:100]")

    generate = TextGeneration(
        llm=OpenAILLM(model="gpt-4o-mini"),
        system_prompt="You are a helpful AI assistant.",
        columns=["instruction"],
    )

    load >> generate

distiset = pipeline.run(use_cache=False)
distiset.push_to_hub("your-username/synthetic-dataset")
'''
print(DISTILABEL_CODE)


# pip install distilabel[openai]
from distilabel.pipeline import Pipeline
from distilabel.steps import LoadDataFromHub
from distilabel.steps.tasks import TextGeneration, SelfInstruct
from distilabel.llms import OpenAILLM

with Pipeline(name="synthetic-data") as pipeline:
    load = LoadDataFromHub(repo_id="argilla/databricks-dolly-15k", split="train[:100]")

    generate = TextGeneration(
        llm=OpenAILLM(model="gpt-4o-mini"),
        system_prompt="You are a helpful AI assistant.",
        columns=["instruction"],
    )

    load >> generate

distiset = pipeline.run(use_cache=False)
distiset.push_to_hub("your-username/synthetic-dataset")



## Additional Learning Resources

### Papers
- [Self-Instruct](https://arxiv.org/abs/2212.10560) Wang et al., 2022
- [Alpaca](https://arxiv.org/abs/2303.03846) Taori et al., 2023
- [Evol-Instruct / WizardLM](https://arxiv.org/abs/2304.12244) Xu et al., 2023
- [Orca](https://arxiv.org/abs/2306.02707) Mukherjee et al., 2023
- [Magpie](https://arxiv.org/abs/2406.08464) Xu et al., 2024
- [Persona Hub](https://arxiv.org/abs/2406.20094) Chan et al., 2024

### Tools
- [distilabel](https://github.com/argilla-io/distilabel)
- [Argilla](https://github.com/argilla-io/argilla) annotation platform
- [LabelStudio](https://labelstud.io/)
- [Hugging Face Datasets](https://huggingface.co/docs/datasets/)